# Explainable AI for Diabetic Retinopathy — Learn & Build

This notebook is written to be run **top to bottom in Google Colab**, one cell at a time.
No local setup needed — just open this file at [colab.research.google.com](https://colab.research.google.com), upload it, and run cells with Shift+Enter.

**Before you start:** Runtime menu -> Change runtime type -> set Hardware accelerator to **GPU** (T4 is fine, free tier). Training on CPU will be painfully slow.

We'll build this in the same order you'd naturally think about the problem:
1. Get the data, understand what we're looking at
2. Teach the computer to look at images (the "Dataset" concept)
3. Build the model (transfer learning, explained plainly)
4. Train it and check if it's actually learning
5. Ask it to explain its own decisions (Grad-CAM)

Nothing here requires prior ML knowledge — just run each cell, read the explanation above it, and see what happens.

## Step 0: Install what we need

`timm` gives us pretrained image models. `grad-cam` gives us the explainability tool. Everything else (`torch`, `pandas`) is already in Colab.

In [ ]:
!pip install -q timm grad-cam

## Step 1: Get the data

We're using **APTOS 2019 Blindness Detection** — a public dataset of real retinal photos, each labeled 0-4 for DR severity. It's on Kaggle.

**To download it into Colab:**
1. Go to kaggle.com -> your profile -> Account -> "Create New API Token". This downloads a file called `kaggle.json`.
2. Run the cell below, and when prompted, upload that `kaggle.json` file.

This is a one-time setup step. If your team already has the dataset in Google Drive, skip this and mount Drive instead (there's a commented-out alternative below).

In [ ]:
from google.colab import files
files.upload()  # upload kaggle.json here

!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

# NOTE: you must accept the competition rules on kaggle.com/c/aptos2019-blindness-detection
# first (click 'Join Competition' / 'I Understand and Accept') or this download will
# fail with a 403 error - this is a one-time step on Kaggle's website.
#
# We download the whole competition bundle at once rather than individual files with
# -f flags - Kaggle's individual-file download endpoint has been unreliable/returns 404
# for some files even after accepting the rules, but the full bundle download works.
!kaggle competitions download -c aptos2019-blindness-detection
!unzip -q aptos2019-blindness-detection.zip -d aptos_data
!ls aptos_data

# Alternative if a teammate shares a Drive folder instead:
# from google.colab import drive
# drive.mount('/content/drive')
# then point DATA_DIR / CSV_PATH below at the Drive paths

**Check the `ls aptos_data` output above** to see the actual folder structure before
continuing - it should show `train.csv`, `train_images`, `test.csv`, `test_images`.
Sometimes it nests one level deeper (e.g. `aptos_data/train_images/train_images/`) -
if so, add that extra folder name to `DATA_DIR` in the next cell.

## Step 2: Look at the data before doing anything else

This is a habit worth building: **always look at your data first.** Let's load the CSV (which maps image filenames to their diagnosis) and view a few actual images with their labels.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import os

DATA_DIR = "aptos_data/train_images"
CSV_PATH = "aptos_data/train.csv"

df = pd.read_csv(CSV_PATH)
print("Total images:", len(df))
print(df.head())
print("\nHow many images per class (0=No DR ... 4=Proliferative DR):")
print(df['diagnosis'].value_counts().sort_index())

CLASS_NAMES = ["No DR", "Mild", "Moderate", "Severe", "Proliferative DR"]

fig, axes = plt.subplots(1, 5, figsize=(15, 3))
for i, cls in enumerate(range(5)):
    row = df[df['diagnosis'] == cls].iloc[0]
    img = Image.open(os.path.join(DATA_DIR, row['id_code'] + '.png'))
    axes[i].imshow(img)
    axes[i].set_title(CLASS_NAMES[cls])
    axes[i].axis('off')
plt.show()

**What you should notice:** the classes are imbalanced (way more "No DR" images than "Proliferative DR"). This matters later — a model can get high accuracy by just always guessing "No DR" and mostly being right, which is not what we want. We'll keep this in mind when judging results, not just trusting one accuracy number.

## Step 3: The Dataset class — teaching PyTorch how to read your data

PyTorch needs your data wrapped in a specific shape. A `Dataset` class just needs to answer two questions when asked:
- `__len__`: "how many items do you have?"
- `__getitem__(i)`: "give me item number i" -> returns (image, label)

That's the whole concept. Everything else in the class below is just the mechanics of opening the file and applying transforms (next step).

In [ ]:
import torch
from torch.utils.data import Dataset

class DRDataset(Dataset):
    def __init__(self, dataframe, img_dir, transform):
        self.df = dataframe.reset_index(drop=True)
        self.img_dir = img_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = os.path.join(self.img_dir, row['id_code'] + '.png')
        img = Image.open(img_path).convert('RGB')
        img = self.transform(img)          # turns the image into numbers the model can use
        label = int(row['diagnosis'])      # the correct answer, 0-4
        return img, label

## Step 4: Transforms — why we distort the training images on purpose

A "transform" converts a photo into a grid of numbers (a **tensor**) the model can process, and resizes everything to the same size (224x224).

We also add small random changes ONLY to the training set — flips, rotation, brightness jitter. This is called **augmentation**. It's on purpose: it stops the model from memorizing exact images and forces it to learn the actual patterns of disease, and it mimics the real-world messiness of smartphone-fundus photos (glare, slight blur) that this problem statement cares about.

The validation set gets NO random changes — we want a clean, honest test of what the model actually learned.

In [ ]:
from torchvision import transforms

IMG_SIZE = 224

train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

val_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

## Step 5: Split into train/validation

We hold back 15% of the data that the model NEVER trains on, so we can honestly check how it does on images it hasn't memorized. `stratify` makes sure both sets have a similar mix of all 5 classes.

In [ ]:
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader

train_df, val_df = train_test_split(df, test_size=0.15, stratify=df['diagnosis'], random_state=42)

train_ds = DRDataset(train_df, DATA_DIR, train_transform)
val_ds = DRDataset(val_df, DATA_DIR, val_transform)

# DataLoader batches items together and shuffles the training set each epoch
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True, num_workers=2)
val_loader = DataLoader(val_ds, batch_size=32, shuffle=False, num_workers=2)

print(f"Train: {len(train_ds)} images | Val: {len(val_ds)} images")

## Step 6: The model — transfer learning, explained plainly

We are NOT building a neural network from scratch — that would need millions of images and weeks of training. Instead we use **transfer learning**:

> Take a model (`EfficientNet-B0`) that already learned to recognize shapes, edges, and textures from 1.4 million general photos (ImageNet). Keep all that learned knowledge, and just replace its final decision layer with a fresh one that outputs 5 numbers (one score per DR grade) instead of whatever it originally predicted.

It's like hiring someone who already knows how to "see" in general, and just teaching them this one new specific task, instead of teaching a baby to see AND diagnose at the same time.

In [ ]:
import timm
import torch.nn as nn

NUM_CLASSES = 5
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

model = timm.create_model("efficientnet_b0", pretrained=True, num_classes=NUM_CLASSES)
model = model.to(device)

## Step 6b: Save checkpoints to Google Drive (do this before training)

**Why this matters:** Colab's local storage is wiped every time your session disconnects — including when you hit a usage limit. If you only save at the very end of training, hitting a limit mid-training means starting completely over. Saving to Google Drive instead means your progress survives disconnects, and you can resume later.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
CHECKPOINT_DIR = "/content/drive/MyDrive/dr_screening_checkpoints"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
CHECKPOINT_PATH = os.path.join(CHECKPOINT_DIR, "dr_effnet_b0.pt")
print("Checkpoints will be saved to:", CHECKPOINT_PATH)


Each **epoch** = one full pass through all training images. For every batch of images:
1. **Forward pass**: show the model the images, it guesses 5 scores per image
2. **Loss**: compare the guess to the true label — a single number saying "how wrong was this"
3. **Backward pass**: figure out which internal numbers (weights) caused the error
4. **Optimizer step**: nudge those weights slightly to reduce the error next time

Repeat thousands of times and the model gradually gets better.

### Fixing the class imbalance problem

Remember Step 2 — there are far more "No DR" images than the other classes. Left alone, the loss function treats every image equally, so the model can lower its overall error just by leaning toward the majority class ("No DR") whenever it's unsure — which is exactly the "showing No DR for DR eyes" behavior you might see.

The fix: tell the loss function to penalize mistakes on rare classes more heavily than mistakes on the common class. This is called **class weighting** — classes with fewer images get a higher weight, so getting them wrong "costs" the model more during training.

In [ ]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

class_counts = train_df['diagnosis'].value_counts().sort_index()
print("Training images per class:\n", class_counts)

weights = compute_class_weight(
    class_weight='balanced',
    classes=np.arange(NUM_CLASSES),
    y=train_df['diagnosis'].values
)
class_weights = torch.tensor(weights, dtype=torch.float32).to(device)
print("\nComputed class weights (higher = rarer class, penalized more):")
for i, w in enumerate(weights):
    print(f"  {CLASS_NAMES[i]}: {w:.2f}")

**About resuming:** the training cell below will detect your existing checkpoint (from the earlier 20-epoch run) and continue from epoch 20 onward — you don't lose that training. The only change is the loss function switches to the weighted version starting now. This isn't identical to training with weighting from epoch 1, but it lets the model correct its bias toward "No DR" over the remaining epochs without discarding what it already learned. If results still look off after this, a fresh run with weighting from the start (delete the checkpoint file in Drive first) would be the cleaner fix, time permitting.

In [ ]:
EPOCHS = 30  # bumped up since we're resuming from epoch 20, not starting over
START_EPOCH = 0

optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)
criterion = nn.CrossEntropyLoss(weight=class_weights)  # now penalizes rare-class mistakes more

# Resume from a previous run if a checkpoint already exists in Drive
if os.path.exists(CHECKPOINT_PATH):
    checkpoint = torch.load(CHECKPOINT_PATH, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    START_EPOCH = checkpoint['epoch'] + 1
    print(f"Resuming from epoch {START_EPOCH} (found existing checkpoint in Drive)")
else:
    print("No existing checkpoint found — starting fresh from epoch 0")

for epoch in range(START_EPOCH, EPOCHS):
    # --- training ---
    model.train()
    running_loss = 0.0
    for imgs, labels in train_loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * imgs.size(0)
    train_loss = running_loss / len(train_ds)

    # --- validation (no learning, just checking) ---
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            preds = torch.argmax(model(imgs), dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    val_acc = correct / total

    print(f"Epoch {epoch+1}/{EPOCHS} | train_loss={train_loss:.4f} | val_acc={val_acc:.4f}")

    # Save after EVERY epoch, not just at the end — this is what protects you
    # from losing progress if you hit Colab's usage limit mid-training.
    torch.save({
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'val_acc': val_acc,
    }, CHECKPOINT_PATH)

print("Training complete. Best/last checkpoint saved to:", CHECKPOINT_PATH)

**If you hit Colab's usage limit partway through:** don't panic, your progress is saved in Drive. Wait for the limit to reset (usually a few hours), reopen this notebook, run Steps 0-6b again (installs + data + model definition + Drive mount), then re-run this training cell — it will automatically detect the existing checkpoint and resume from where it stopped instead of starting over.

**One extra step needed for the web app:** the checkpoint now saves a dictionary with `model_state_dict` inside it (plus training info), not just the raw weights like before. Run the cell below once training is done — it extracts just the model weights into a separate file that matches what `backend/app.py` expects.

In [ ]:
# Extract just the model weights for the web app (app.py expects the raw
# state_dict, not the full checkpoint dict with optimizer/epoch info)
checkpoint = torch.load(CHECKPOINT_PATH, map_location=device)
torch.save(checkpoint['model_state_dict'], "dr_effnet_b0.pt")
print("Saved app-ready weights to dr_effnet_b0.pt — download this one from the Colab file panel.")

**What to look for:** `train_loss` should generally go down each epoch. `val_acc` should generally go up. Push toward 15-20 epochs for a reasonably confident model — this dataset commonly reaches somewhere around 65-80% validation accuracy with this setup.

## Step 8: Explainability — Grad-CAM, and why judges will care about this specifically

Right now the model gives you a grade with no reasoning. Grad-CAM answers: **"which part of the image made you say that?"**

How it works (plain version): during the forward pass, the model builds up an internal map of "important regions" in its last convolutional layer before making a decision. Grad-CAM looks at how much each region contributed to the final answer and turns that into a heatmap you can overlay on the original photo — red/yellow = "this mattered a lot", blue = "this didn't matter".

Let's run it on one validation image and see it visually.

In [ ]:
import numpy as np
from pytorch_grad_cam import GradCAMPlusPlus
from pytorch_grad_cam.utils.image import show_cam_on_image
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget

# grab one image from the validation set to explain
sample_img_tensor, true_label = val_ds[0]
input_tensor = sample_img_tensor.unsqueeze(0).to(device)

model.eval()
with torch.no_grad():
    probs = torch.softmax(model(input_tensor), dim=1)[0]
    predicted_class = int(torch.argmax(probs).item())

print(f"True label: {CLASS_NAMES[true_label]} | Model predicted: {CLASS_NAMES[predicted_class]} ({probs[predicted_class]*100:.1f}% confident)")

# Grad-CAM needs the un-normalized 0-1 image to draw the overlay on
row = val_df.iloc[0]
raw_img = Image.open(os.path.join(DATA_DIR, row['id_code'] + '.png')).convert('RGB').resize((IMG_SIZE, IMG_SIZE))
rgb_float = np.array(raw_img).astype(np.float32) / 255.0

target_layers = [model.conv_head]  # last conv block of EfficientNet-B0
cam = GradCAMPlusPlus(model=model, target_layers=target_layers)
grayscale_cam = cam(input_tensor=input_tensor, targets=[ClassifierOutputTarget(predicted_class)])[0]
overlay = show_cam_on_image(rgb_float, grayscale_cam, use_rgb=True)

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(raw_img); axes[0].set_title("Original"); axes[0].axis('off')
axes[1].imshow(overlay); axes[1].set_title(f"Grad-CAM: {CLASS_NAMES[predicted_class]}"); axes[1].axis('off')
plt.show()

## Where to go from here

1. **Re-run Step 7 with more epochs** (15-20) once you've confirmed the loop works — this notebook used 5 just to move fast.
2. Download `dr_effnet_b0.pt` from Colab's file panel (left sidebar) — this is your trained model.
3. Drop it into the `backend/checkpoints/` folder of the web app project, point `WEIGHTS_PATH` in `app.py` at it, and run the FastAPI + `index.html` demo — that part doesn't require retraining anything, it just loads what you made here.
4. Once this baseline works, look at IDRiD's lesion masks to make the Grad-CAM explanation more precise (mentioned in the project README) — that's the kind of detail that separates a good submission from a great one.